## 로컬 환경 설정

로컬 Jupyter나 VS Code에서 실행할 때 현재 경로의 부모를 탐색해 저장소 루트를 찾는다. 로컬 dependency는 notebook 설치 셀보다 `uv sync`로 맞춘 환경을 우선한다.


In [1]:
import os
import sys
from pathlib import Path

# 현재 notebook 위치: VLM/model
# 실제 프로젝트 루트: VLM
LOCAL_PROJECT_PATH = Path.cwd().parent

os.chdir(LOCAL_PROJECT_PATH)

if str(LOCAL_PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(LOCAL_PROJECT_PATH))

print("Project root:", LOCAL_PROJECT_PATH)


Project root: /home/undergraduate/20231372_TY/Pytorch/VLM


## GPU 확인

**현재 런타임의 GPU와 PyTorch CUDA 인식 상태를 확인한다.** Colab에서는 환경 설정과 패키지 설치 전후에 한 번 확인하면 된다.


In [2]:
# GPU 확인
# LoRA fine-tuning은 GPU 실행을 전제로 하므로 nvidia-smi와 공용 device helper로 CUDA 상태를 확인합니다.
# get_device_info 결과는 torch가 실제로 GPU를 인식하는지 확인하는 기준입니다.
import subprocess

# device_utils: CUDA/CPU device 선택과 memory 정리를 공용 helper로 처리합니다.
from utils.device_utils import get_device_info

try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi 명령을 찾을 수 없습니다. GPU 런타임인지 확인하세요.')

gpu_info = get_device_info()
print('CUDA available:', gpu_info['cuda_available'])
if gpu_info['cuda_available']:
    print('GPU:', gpu_info['device_name'])


Mon Jun  1 01:07:37 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.230.02             Driver Version: 535.230.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA TITAN Xp                Off | 00000000:19:00.0 Off |                  N/A |
| 23%   29C    P8               9W / 250W |     11MiB / 12288MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## 공통 import

노트북 전체에서 사용하는 외부 패키지와 저장소 공용 모듈을 한 번만 불러온다.


## COCO 데이터셋이란
COCO(Common Objects in Context)는 이미지와 그에 대한 캡션이 쌍으로 묶인 대표적인 데이터셋이다.

```
이미지: 강아지가 공원에서 뛰는 사진
캡션:  "A dog is running in the park"
       "A brown dog plays outdoors"
       "A pet running on grass"
       ...  ← 이미지 하나에 캡션 여러 개
```


> bundle이라는 이름의 의미

여러 개를 한 묶음으로 만들어 반환한다는 뜻

```
bundle = prepare_coco_caption_dataset_bundle(...)

bundle["train"]  # 학습용 데이터
bundle["eval"]   # 평가용 데이터
bundle["test"]   # 테스트용 데이터
```






In [3]:
import os
import json
from dataclasses import dataclass
from pathlib import Path

MPLCONFIGDIR = Path(os.environ.get("MPLCONFIGDIR", "/tmp/matplotlib"))
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import torch
import transformers
from PIL import Image
from torch.utils.data import Dataset

from utils.caption_metrics import compute_caption_metrics

from utils.device_utils import (
    get_device_info,
    release_cuda_memory,
    resolve_device,
)

from utils.logger_utils import configure_logger, get_logger

from utils.smolvlm_utils import (
    align_model_generation_config_with_tokenizer,
    generate_caption_splits,
    load_model_and_processor,
    resolve_auto_model_class,
    resolve_torch_dtype,
)

from utils.training_utils import (
    build_config_payload,
    save_json,
)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

/home/undergraduate/20231372_TY/envs/smolvlm_lora/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Section 1: COCO LoRA Fine-tuning

COCO captioning subset에 LoRA fine-tuning을 적용하고 zero-shot/full fine-tuning/LoRA의 장단점을 정리한다.


## 실행 설정과 상수

모델, prompt, output 경로, LoRA target module, Trainer hyperparameter를 고정한다. 기본 실행은 rank 4/8/16 sweep으로 LoRA adapter 크기에 따른 비용과 성능 변화를 비교한다.


In [4]:
# 실행 설정과 상수
# COCO가 아니라 NurViD에서 추출한 의료 이미지 + 직접 작성한 GT caption을 사용한다.

LOGGER = get_logger("nurvid_smolvlm_lora")

@dataclass(slots=True)
class LoRAFinetuneConfig:
    # 사용할 SmolVLM 모델
    model_name: str = "HuggingFaceTB/SmolVLM-256M-Instruct"

    # baseline / fine-tuned 모델 모두 같은 prompt로 caption 생성
    prompt: str = "Describe this medical scene in one concise English sentence."

    # 현재 데이터가 작으므로 rank는 하나만 먼저 실험
    ranks: tuple[int, ...] = (8,)

    # LoRA를 적용할 모듈
    target_modules: tuple[str, ...] = (
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    )

    seed: int = 42
    device: str = "cuda"

    # 네 GPU는 TITAN Xp 12GB라서 batch size를 작게 시작하는 게 안전함
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 4

    # 데이터가 작으므로 일단 1 epoch로 파이프라인 확인
    num_train_epochs: int = 1
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    logging_steps: int = 1
    max_new_tokens: int = 48

    # NurViD 이미지와 caption 경로
    image_dir: Path = Path("dataset/selected_images")
    caption_path: Path = Path("dataset/nurvid_captions.jsonl")

    # 산출물 저장 경로
    data_dir: Path = Path("data/nurvid-lora-finetuning")
    output_dir: Path = Path("model/nurvid-lora-finetuning")

In [5]:
config_lora_ft = LoRAFinetuneConfig()

config_lora_ft.data_dir.mkdir(parents=True, exist_ok=True)
config_lora_ft.output_dir.mkdir(parents=True, exist_ok=True)

print("Image dir:", config_lora_ft.image_dir)
print("Caption path:", config_lora_ft.caption_path)
print("Output dir:", config_lora_ft.output_dir)

Image dir: dataset/selected_images
Caption path: dataset/nurvid_captions.jsonl
Output dir: model/nurvid-lora-finetuning


## LoRA 학습 준비

LoRA fine-tuning 설정, device 정보, 저장 경로를 준비한다. = 학습을 시작하기 전 환경을 준비하고 설정을 기록하는 코드


In [6]:
# LoRA 학습 준비

configure_logger()
config_lora_ft = LoRAFinetuneConfig()

# 디렉토리 생성
config_lora_ft.data_dir.mkdir(parents=True, exist_ok=True)
config_lora_ft.output_dir.mkdir(parents=True, exist_ok=True)

# 이미지 / 캡션 파일 확인
image_files = sorted([
    p for p in config_lora_ft.image_dir.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

if not config_lora_ft.image_dir.exists():
    raise FileNotFoundError(f"이미지 폴더가 없습니다: {config_lora_ft.image_dir}")

if not config_lora_ft.caption_path.exists():
    raise FileNotFoundError(f"캡션 파일이 없습니다: {config_lora_ft.caption_path}")

LOGGER.info(
    "Starting NurViD LoRA run | model=%s images=%s ranks=%s output_dir=%s",
    config_lora_ft.model_name,
    len(image_files),
    config_lora_ft.ranks,
    config_lora_ft.output_dir,
)

# GPU 확인
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 필요합니다. 현재 torch.cuda.is_available() = False 입니다.")

device = resolve_device(config_lora_ft.device)

device_info = get_device_info(str(device))
save_json(
    config_lora_ft.output_dir / "device_info.json",
    {
        **device_info,
        "device": str(device_info["device"]),
    },
)

print("Device:", device)
print("Image count:", len(image_files))
print("Caption file:", config_lora_ft.caption_path)

01:07:52 |   22 | INFO     | Starting NurViD LoRA run | model=HuggingFaceTB/SmolVLM-256M-Instruct images=7 ranks=(8,) output_dir=model/nurvid-lora-finetuning
Device: cuda
Image count: 7
Caption file: dataset/nurvid_captions.jsonl


In [8]:
# selected_images 안의 이미지 목록으로 caption 템플릿 만들기

caption_path = Path("dataset/nurvid_captions.jsonl")
image_dir = Path("dataset/selected_images")

image_files = sorted([
    p.name for p in image_dir.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

with open(caption_path, "w", encoding="utf-8") as f:
    for image_name in image_files:
        row = {
            "image": image_name,
            "caption": "A nurse performing a medical action in a hospital setting."
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("created:", caption_path)
print("image count:", len(image_files))

created: dataset/nurvid_captions.jsonl
image count: 7


## 데이터 split 준비

COCO train/val/test split과 caption 평가 sample을 준비한다.

> config_lora_ft 안의 설정값들

```
config_lora_ft.train_images = 1000  → 학습용 1000개
config_lora_ft.val_images   = 20    → 검증용 20개
config_lora_ft.test_images  = 50    → 테스트용 50개
config_lora_ft.compare_images = 4   → 비교용 4개
```

반환되는 dataset_bundle은 딕셔너리이다.


```
dataset_bundle = {
    "train_rows": [...],         # 학습용 1000개
    "val_rows": [...],           # 검증용 20개
    "metric_samples": [...],     # 점수 계산용 50개
    "comparison_samples": [...], # 눈으로 비교용 4개
    "manifest": {...},           # 어떻게 나눴는지 기록
}
```



> comparison_samples


```
comparison_samples = dataset_bundle["comparison_samples"]
```

4개의 샘플로 다음과 같은 비교를 한다.

```
이미지 1개당
Zero-shot 캡션  : "A dog is playing"
Full FT 캡션    : "A brown dog runs on grass"
LoRA 캡션       : "A dog running in a park"
정답 캡션       : "A dog is running in the park"
```




> 전체 목적

LoRA 학습에 필요한 데이터를 4가지 용도로 나눈다.

```
전체 COCO 데이터
        ↓
prepare_coco_caption_dataset_bundle
        ↓
train_rows        1000개  ← 학습용
val_rows            20개  ← 검증용
metric_samples      50개  ← 점수 계산용
comparison_samples   4개  ← 눈으로 비교용
```








In [8]:
# 데이터 split 준비

# 공용 COCO helper가 train/validation/metric/comparison split을 만들고 split_manifest.json으로 저장할 metadata를 반환합니다.
# train_rows와 val_rows는 rank별 LoRA Trainer가 사용할 row 단위 학습 입력입니다.

# config_lora_ft = LoRAFinetuneConfig()
# config_lora_ft 안의 설정값들을 보고 데이터를 나눈다.
dataset_bundle = prepare_coco_caption_dataset_bundle(config_lora_ft, logger=LOGGER)

# zero-shot, Full FT, LoRA 결과를 눈으로 나란히 비교할 4개의 샘플
comparison_samples = dataset_bundle["comparison_samples"]

# metric_samples : 학습이 끝난 후 BLEU, ROUGE 같은 점수를 계산할 50개 샘플
# 4개는 너무 적으므로 50개로 더 정확하게 측정한다.
metric_samples = dataset_bundle["metric_samples"]

# manifest 저장, 어떤 이미지가 어느 split에 들어갔는지 파일로 저장한다.
# 나중에 같은 데이터로 재현할 수 있도록 기록
'''
  data/2-5-coco-lora-finetuning/split_manifest.json
  {
      "train":      [이미지 id 1000개],
      "val":        [이미지 id 20개],
      "metric":     [이미지 id 50개],
      "comparison": [이미지 id 4개]
  }
'''
save_json(
    config_lora_ft.data_dir / "split_manifest.json",
    dataset_bundle["manifest"],
)

# 실제 LoRA 학습에 쓰이는 데이터
'''
각 row의 형태
  {
      "image": 이미지 데이터,
      "caption": "A dog is running in the park"
  }
'''
train_rows = dataset_bundle["train_rows"]
val_rows = dataset_bundle["val_rows"]


NameError: name 'prepare_coco_caption_dataset_bundle' is not defined

## zero-shot 추론 준비

LoRA 학습 전 baseline caption을 만들 base SmolVLM model과 processor를 준비한다.

> 정밀도란

모델의 가중치를 저장하는 숫자의 정밀도

```
FP32  → 소수점 32비트로 저장  (정밀, 메모리 많이 씀)
FP16  → 소수점 16비트로 저장  (메모리 절반, GPU에서 빠름)
BF16  → 소수점 16비트로 저장  (FP16보다 안정적, 최신 GPU)
```



> resolve_auto_model_class

Hugging Face에서 모델을 로드할 때 모델 종류에 맞는 클래스를 사용해야 한다.

```
텍스트만 처리하는 모델  → AutoModelForCausalLM
이미지+텍스트 모델      → AutoModelForImageTextToText
번역 모델              → AutoModelForSeq2SeqLM
```
SmolVLM은 이미지와 텍스트를 둘 다 처리하므로 AutoModelForImageTextToText가 필요하다



> 왜 zero-shot이 필요한가?

```
zero-shot 성능  ← 아무것도 안 배운 상태
LoRA rank=4 성능
LoRA rank=8 성능
LoRA rank=16 성능
Full FT 성능    ← 전체 학습한 상태
```
이렇게 비교해야 LoRA가 얼마나 효과적인지 알 수 있다.







In [ ]:
# zero-shot 추론 준비(LoRA 학습 전 기준점을 만들기 위해 모델을 로드하는 코드)

# zero-shot baseline 생성을 위해 base SmolVLM과 processor를 먼저 로드합니다.
# auto_model_class와 torch_dtype는 이후 rank별 LoRA 학습 helper가 같은 model class와 dtype 정책을 재사용하도록 보관합니다.

# 이 gpu에서 모델을 어떤 정밀도로 올릴지 자동으로 결정
# auto로 두면 T4에 맞게 자동으로 FP16을 선택
torch_dtype = resolve_torch_dtype(device=device, torch_dtype="auto")

# SmolVLM 같은 멀티모달 모델을 로드할 때 어떤 클래스를 써야 하는지 자동으로 찾아주는 코드
# SmolVLM은 이미지 + 텍스트를 처리하는 멀티모달이므로 AutoModelForImageTextToText
# 이 값을 저장해두는 이유는 나중에 rank별 LoRA 학습할 때도 같은 클래스를 재사용하기 위함
auto_model_class = resolve_auto_model_class(transformers)

# SmolVLM 모델과 프로세서를 한 번에 로드하는 함수
zero_shot_bundle = load_model_and_processor(
    model_id=config_lora_ft.model_name, # 어떤 모델을 불러올지 지정(SmolVLM-256M-Instruct)
    device=device, # 모델을 어디에 올릴지 지정
    torch_dtype=torch_dtype, # FP16
    prefer_flash_attention=False, # FlashAttention은 Attention 연산을 더 빠르게 하는 기술인데, T4에서는 지원되지 않아 끄는 것
)
'''
반환값 : 모델과 프로세서를 딕셔너리로 묶어서 반환해준다.
  zero_shot_bundle = {
      "model":     실제 신경망,이미지를 보고 캡션을 생성
      "processor": 이미지/텍스트를 모델 입력 형식으로 변환
  }
'''

# 반환값에서 각각 꺼낸다
processor = zero_shot_bundle["processor"]
zero_shot_model = zero_shot_bundle["model"]

# 모델의 생성 설정과 토크나이저를 서로 일치시키는 코드
# 모델과 토크나이저의 문장 끝 토큰을 일치시킨다.

'''
맞추기 전                    맞춘 후
─────────────   ──────────────
모델:  eos = 2               모델:  eos = 32000
토크나이저: eos = 32000  →   토크나이저: eos = 32000
→ 캡션이 안 끝남              → 캡션이 정상 종료
'''

align_model_generation_config_with_tokenizer(zero_shot_model, processor.tokenizer)


## zero-shot 추론 실행과 평가

LoRA 학습 전 기준 caption과 metric을 만든다. = 아래의 코드는 아무것도 학습하지 않은 상태에서 모델이 캡션을 얼마나 잘 생성하는지 측정하는 코드

> 점수 계산 zero_shot_scores

```
모델 생성: "A dog is running in a field"
정답:      "A brown dog runs on the grass"
        ↓
BLEU   = 0.32
ROUGE  = 0.45
METEOR = 0.38
CIDEr  = 0.71
```

safe가 붙은 이유는 점수 계산 중 오류가 나도 프로그램이 멈추지 않게 처리했기 때문이다



In [ ]:
# zero-shot 추론 실행과 평가

# 학습 전 base model로 comparison split과 metric split의 zero-shot caption을 생성합니다.
# comparison prediction은 qualitative report에, metric prediction은 BLEU/METEOR/CIDEr-D 계산에 사용합니다.
# zero_shot_caption_scores.json은 최종 rank summary와 method comparison의 기준 row입니다.

# 이미지들을 모델에 넣어서 캡션을 생성한다.
zero_shot_outputs = generate_caption_splits(
    label="Zero-shot", # 로그에 "Zero-shot"으로 표시
    model=zero_shot_model, # 캡션을 생성할 모델
    processor=processor, # 이미지를 모델 입력으로 변환
    sample_splits={
        "comparison": comparison_samples, # 4개 -> 비교용
        "metric": metric_samples, # 최대 48토큰
    },
    device=device,
    prompt=config_lora_ft.prompt, # "Describe this image is one concise English sentence"
    max_new_tokens=config_lora_ft.max_new_tokens, # 최대 48토큰까지만 생성
    batch_size=config_lora_ft.per_device_eval_batch_size, # 한 번에 몇 개씩 처리할지
    logger=LOGGER,
)

# 결과 꺼내기
zero_shot_comparison_predictions = zero_shot_outputs["comparison"] # 4개 캡션(나중에 눈으로 비교하기 위해)
zero_shot_metric_predictions = zero_shot_outputs["metric"] # 50개 캡션 (나중에 점수로 계산하기 위해)
'''
  zero_shot_metric_predictions = [
      "A dog is running in a field",
      "A cat sitting on a chair",
      "Two people walking on a street",
      ...  # 50개
  ]
'''

# 점수 계산
zero_shot_scores = safe_compute_single_caption_scores(
    predictions=zero_shot_metric_predictions, # 모델이 생성한 캡션
    samples=metric_samples, # 정답 캡션
    logger=LOGGER,
)

# 점수 저장, 계산된 점수를 파일로 저장한다.
# 나중에 LoRA 학습 후 점수와 비교할 기준점으로 사용
save_json(config_lora_ft.output_dir / "zero_shot_caption_scores.json", zero_shot_scores)
'''
model/2-5-coco-lora-finetuning/zero_shot_caption_scores.json
{
    "bleu": 0.32,
    "rouge": 0.45,
    "meteor": 0.38,
    "cider": 0.71
}
'''



## zero-shot model 메모리 정리

baseline caption을 만든 뒤 base model 메모리를 비워 rank별 LoRA 학습 공간을 확보한다.

 = 더 이상 필요없는 zero-shot 모델을 GPU 메모리에서 제거하는 코드이다. (그냥 두면 GPU 메모리를 계속 차지하기 때문)


In [ ]:
# zero-shot model 메모리 정리

# zero-shot model은 기준 caption 생성이 끝나면 더 이상 필요하지 않으므로 GPU memory에서 제거합니다.
# processor는 rank별 평가에도 필요하므로 model 객체만 삭제하고 processor는 유지합니다.
del zero_shot_model # 파이썬 변수에서 모델을 삭제
del zero_shot_bundle["model"] # 딕셔너리 안에도 모델 참조가 남아있으므로, 거기에서도 삭제
release_cuda_memory(device, torch) # 파이썬 변수에서 지웠다고 GPU 메모리가 바로 비워지지 않는다. 이 함수 실행해야 GPU 메모리 실제로 정리

## LoRA 학습 실행

설정된 rank 목록을 순회하며 LoRA 학습, 평가, artifact 저장을 실행한다. 기본값은 rank 4/8/16 sweep이다.


In [ ]:
# LoRA 학습 실행

# config에 정의된 rank마다 독립 LoRA adapter를 학습하고 평가합니다.
# train_and_evaluate_lora_rank helper는 base model 로드, PEFT 적용, Trainer 학습, caption 생성, metric 계산, artifact 저장을 rank 단위로 수행합니다.
# rank_summary_rows는 그래프/table용 요약이고 rank_results는 method comparison report에 들어갈 상세 결과입니다.

# 결과를 담을 빈 리스트를 만든다.
rank_summary_rows = [] # 그래프/표용 요약 결과
rank_results = [] # 상세 결과
full_finetuning_trainable_params = None # 나중에 Full FT 파라미터 수를 저장할 변수를 미리 만들어두는 것

# zero-shot 모델 메모리 해제
# zero-shot 모델을 지워서 LoRA 학습을 위한 GPU 메모리 확보
del zero_shot_model
torch.cuda.empty_cache()
release_cuda_memory()

# rank loop는 같은 데이터 split과 prompt를 유지한 채 adapter capacity만 바꿔 비교합니다.
for rank in config_lora_ft.ranks:
    # helper 내부에서 rank별 output_dir, PEFT config, Trainer, caption 평가가 모두 처리됩니다.
    rank_result = train_and_evaluate_lora_rank(
        rank=int(rank), # 현재 rank (4,8,16 중 하나)
        config=config_lora_ft, # 모든 설정값
        processor=processor, # 입출력 변환기
        auto_model_class=auto_model_class, # 모델 클래스
        transformers_module=transformers, # transformers 라이브러리
        torch_module=torch, # torch 라이브러리
        torch_dtype=torch_dtype, # FP16
        device=device, # cuda
        train_rows=train_rows, # 학습 데이터 1000개
        val_rows=val_rows, # 검증 데이터 20개
        comparison_samples=comparison_samples, # 비교용 4개
        metric_samples=metric_samples, # 점수 계산용 50개
        zero_shot_comparison_predictions=zero_shot_comparison_predictions, # zero-shot 비교 캡션
        zero_shot_metric_predictions=zero_shot_metric_predictions, # zero-shot 점수 캡션
        full_finetuning_trainable_params=full_finetuning_trainable_params, # Full FT 파라미터 수
        logger=LOGGER,
    )

    # full FT parameter 수는 rank와 무관하므로 첫 계산값을 다음 rank에 재사용합니다.
    full_finetuning_trainable_params = rank_result["full_finetuning_trainable_params"]

    # 결과 저장
    rank_summary_rows.append(rank_result["summary"])
    rank_results.append(rank_result)

    '''
      루프가 종료되면
      rank_summary_rows = [
      rank=4  요약,
      rank=8  요약,
      rank=16 요약,
      ]

      rank_results = [
          rank=4  상세 결과,
          rank=8  상세 결과,
          rank=16 상세 결과,
      ]
    '''


## rank summary와 method comparison 저장

LoRA rank별 결과와 `Zero-shot / Full FT / LoRA` 비교 table을 함께 저장한다.

- full fine-tuning 산출물이 없으면 skip reason을 남기고 LoRA 결과는 그대로 저장한다.





> rank summary와 method comparizon 저장 코드

이 코드는 모든 학습 결과를 한 곳에 모아서 정리하고 저장하는 코드이다.

> rank_summary, method_comparison

```
rank_summary.json      → 모든 정보 (종합)
method_comparison.json → 비교 결과만 (간단)
```







In [ ]:
# rank summary와 method comparison 저장

# rank별 LoRA 결과와 1.5주차 full fine-tuning 결과를 함께 읽어 method comparison 구조를 만듭니다.
# rank_summary.json은 config, zero-shot 점수, full FT 참조, LoRA rank 요약, comparison report를 모두 담는 종합 artifact입니다.
# method_comparison.json은 후속 문서나 분석에서 바로 읽기 쉬운 비교 전용 artifact입니다.
# 1.5주차 산출물이 없으면 unavailable 상태로 두고 LoRA 결과 저장은 계속합니다.

# Full FT 결과 불러오기
full_finetuning_comparison = load_full_finetuning_comparison(config_lora_ft.full_finetuning_comparison_path)

# table row는 zero-shot, full FT, 각 LoRA rank를 같은 column으로 비교하기 위한 요약입니다.
method_comparison_rows = build_method_comparison_rows(
    zero_shot_scores=zero_shot_scores, # zero-shot 점수
    full_finetuning=full_finetuning_comparison, # Full FT 점수
    lora_rows=rank_summary_rows, # LoRA rank별 점수
    logger=LOGGER,
)
'''
  method_comparison_rows = [
      {"method": "zero-shot", "bleu": 0.21, "rouge": 0.31, ...},
      {"method": "full_ft",   "bleu": 0.45, "rouge": 0.52, ...},
      {"method": "lora_r4",   "bleu": 0.38, "rouge": 0.44, ...},
      {"method": "lora_r8",   "bleu": 0.41, "rouge": 0.48, ...},
      {"method": "lora_r16",  "bleu": 0.43, "rouge": 0.50, ...},
  ]
'''

# 이미지별로 각 방법의 캡션을 나란히 볼 수 있는 리포트를 만든다.
method_comparison_report = build_lora_method_comparison_report(
    samples=comparison_samples, # 비교용 4개 이미지
    zero_shot_predictions=zero_shot_comparison_predictions, # zero-shot 캡션
    zero_shot_scores=zero_shot_scores, # zero-shot 점수
    rank_results=rank_results, # LoRA rank별 결과
    full_finetuning=full_finetuning_comparison, # Full FT 결과
)

'''
  이미지 1:
    zero-shot : "A dog is playing"
    full_ft   : "A brown dog runs on grass"
    lora_r4   : "A dog running in a park"
    lora_r8   : "A dog is running on green grass"
    lora_r16  : "A brown dog is running on the grass"
    정답       : "A dog is running in the park"
'''

# dataclass와 Path 값을 JSON으로 저장 가능한 dict로 변환합니다.
# build_config_payload가 이것들을 JSON이 이해할 수 있는 형태로 변환해준다.
config_payload = build_config_payload(config_lora_ft)

# rank_summary.json 저장 : 모든 정보를 담은 종합 파일, 나중에 이 파일 하나만 읽어도 전체 실험 결과를 파악할 수 있다.
save_json(
    config_lora_ft.output_dir / "rank_summary.json",
    {
        "config": config_payload, # 학습 설정
        "zero_shot_caption_scores": zero_shot_scores, # zero-shot 점수
        "full_finetuning_comparison": full_finetuning_comparison, # Full FT 점수
        "rows": rank_summary_rows, # rank별 요약
        "method_comparison_rows": method_comparison_rows, # 비교 행
        "method_comparison_report": method_comparison_report, # 상세 리포트
    },
)

# 비교 결과만 따로 저장한다. rank_summary.json보다 가볍고 간단하다
save_json(config_lora_ft.output_dir / "method_comparison.json", method_comparison_report)

# 결과를 터미널에 표로 출력한다.
log_rank_summary(rank_summary_rows, LOGGER)
log_method_comparison_rows(method_comparison_rows, LOGGER)
'''
  rank  bleu   rouge  meteor
  ──  ──   ──   ───
  4     0.38   0.44   0.35
  8     0.41   0.48   0.38
  16    0.43   0.50   0.40
'''

# 마지막 시각화 셀과 후속 분석에서 재사용할 대표 결과 객체입니다.
# 나중에 시각화 셀에서 이 딕셔너리 하나만 넘기면 모든 결과를 쓸 수 있도록 한 곳에 묶어둔다.
result_lora_ft = {
    "config": config_payload,
    "zero_shot_caption_scores": zero_shot_scores,
    "full_finetuning_comparison": full_finetuning_comparison,
    "rows": rank_summary_rows,
    "method_comparison_rows": method_comparison_rows,
    "method_comparison_report": method_comparison_report,
}


## LoRA fine-tuning 결과 시각화

`show_lora_rank_summary`로 zero-shot, full fine-tuning, rank별 LoRA 결과를 같은 metric 축에서 비교한다. full fine-tuning 산출물이 없거나 metric을 읽지 못한 경우에는 해당 항목을 표시하지 않는다.

* 아래 코드는 zero-shot, Full FT, LoRA Rank별 결과를 그래프로 그리는 코드이다.



> zero-shot 점수 꺼낼때


```
계산 성공 → zero_shot_plot_metrics = {"bleu": 0.21, ...}  ← 그래프에 표시
계산 실패 → zero_shot_plot_metrics = None                 ← 그래프에서 제외
```




In [ ]:
# LoRA fine-tuning 결과 시각화
# 시각화 전에 zero-shot과 full fine-tuning metric이 실제로 사용 가능한지 확인해 None으로 정리합니다.
# show_lora_rank_summary는 zero-shot, full FT, rank별 LoRA를 같은 metric 축에서 비교합니다.

# zero-shot 점수 준비
zero_shot_plot_metrics = result_lora_ft["zero_shot_caption_scores"]
# metric 계산이 실패해 available=False이면 그래프 입력에서 제외합니다.
if isinstance(zero_shot_plot_metrics, dict) and zero_shot_plot_metrics.get("available") is False:
    zero_shot_plot_metrics = None

# Full FT 점수 준비
# 1.5주차에 학습해둔 Full FT 결과가 있으면 가져오고, 없으면 None으로 두는 코드
full_ft_plot_metrics = None # 결과가 없을 때를 기본값으로 설정한다.
full_ft_report = result_lora_ft.get("full_finetuning_comparison") # result_lora_ft 딕셔너리에서 Full FT 결과 꺼내기

# full FT report가 있고 metric이 유효할 때만 full FT bar를 추가합니다.
if isinstance(full_ft_report, dict) and full_ft_report.get("available"):
    # get("metrics", {})에서 {}는 "metrics"키가 없을 때 빈 딕셔너리를 반환한다는 뜻이다. (오류 방지)
    full_ft_metrics = full_ft_report.get("metrics", {}).get("full_finetuned") # 딕셔너리 안으로 들어가서 실제 점수를 꺼낸다.
    '''
      full_ft_report = {
      "available": True,
      "metrics": {
          "full_finetuned": {       ← 여기서 꺼냄
              "bleu": 0.45,
              "rouge": 0.52,
              ...
              }
          }
      }
    '''
    # 두 번째 확인
    # .get("available", True)에서 True는 "available" 키가 없을 때 기본값으로 True를 쓴다는 뜻
    if isinstance(full_ft_metrics, dict) and full_ft_metrics.get("available", True) is not False:
        full_ft_plot_metrics = full_ft_metrics

# 그래프 그리기
show_lora_rank_summary(
    result_lora_ft["rows"], # LoRA rank별 결과
    zero_shot_metrics=zero_shot_plot_metrics, # zero-shot 점수
    full_finetuning_metrics=full_ft_plot_metrics, # Full FT 점수
)
